# CIS6005 Computational Intelligence
## Notebook 04 — Data Cleaning
**Student Health Risk Prediction | Kaggle PS S6E7**

---
### What Is Data Cleaning?
Data cleaning fixes the problems identified in EDA:
- Remove or impute missing values
- Remove duplicate rows
- Handle or cap outliers
- Fix inconsistent categorical values

> **Golden Rule:** Always clean `train` and `test` with **identical** transformations. The test set must be processed the same way — otherwise your model will predict on different data than it was trained on.

In [1]:
# ============================================================
# IMPORTS & SETUP
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA     = PROJECT_ROOT / 'data' / 'raw'
PROC_DATA    = PROJECT_ROOT / 'data' / 'processed'

# Load raw data (never modify these)
train_raw = pd.read_csv(RAW_DATA / 'train.csv')
test_raw  = pd.read_csv(RAW_DATA / 'test.csv')

# Working copies
train_df = train_raw.copy()
test_df  = test_raw.copy()

print(f'✅ Loaded: Train {train_df.shape} | Test {test_df.shape}')

✅ Loaded: Train (690088, 15) | Test (295753, 14)


## 1. Pre-Cleaning Snapshot

**Why:** We record the state BEFORE cleaning, so we can compare AFTER. This shows the examiner that you followed a systematic process.

In [2]:
# ============================================================
# SECTION 1: Pre-Cleaning Snapshot
# ============================================================

def snapshot(df, name):
    """Print a quick state snapshot of a DataFrame."""
    print(f'  [{name}]  Rows: {len(df):,}  |  '
          f'Missing: {df.isnull().sum().sum():,}  |  '
          f'Duplicates: {df.duplicated().sum():,}')

print('=' * 60)
print('  PRE-CLEANING STATE')
print('=' * 60)
snapshot(train_df, 'Train')
snapshot(test_df,  'Test ')
print('=' * 60)

  PRE-CLEANING STATE


  [Train]  Rows: 690,088  |  Missing: 449,496  |  Duplicates: 0
  [Test ]  Rows: 295,753  |  Missing: 192,642  |  Duplicates: 0


## 2. Remove Duplicate Rows

**Why:** Duplicate rows make the model see the same sample multiple times, creating artificial confidence in those patterns.

**How:** `drop_duplicates()` keeps only the first occurrence of any repeated row.

In [3]:
# ============================================================
# SECTION 2: Remove Duplicates
# ============================================================

before_train = len(train_df)
before_test  = len(test_df)

train_df = train_df.drop_duplicates()
test_df  = test_df.drop_duplicates()

removed_train = before_train - len(train_df)
removed_test  = before_test  - len(test_df)

print('=' * 60)
print('  Duplicate Removal')
print('=' * 60)
print(f'  Train: {removed_train} duplicates removed  → {len(train_df):,} rows remain')
print(f'  Test : {removed_test} duplicates removed   → {len(test_df):,} rows remain')
print('=' * 60)

  Duplicate Removal
  Train: 0 duplicates removed  → 690,088 rows remain
  Test : 0 duplicates removed   → 295,753 rows remain


## 3. Handle Missing Values

**Why:** We must choose an imputation strategy based on the column's data type and distribution:

| Strategy | When to Use |
|----------|-------------|
| **Median imputation** | Numerical with outliers (robust to skew) |
| **Mean imputation** | Numerical with normal distribution, no extreme outliers |
| **Mode imputation** | Categorical columns |
| **Drop rows** | When very few rows (<1%) are missing |

**IMPORTANT:** We compute imputation statistics from **train only**, then apply to both train and test. This prevents data leakage.

In [4]:
# ============================================================
# SECTION 3: Missing Value Imputation
# Strategy computed on TRAIN, applied to BOTH
# ============================================================

numerical_cols   = [c for c in train_df.select_dtypes(include=['int64','float64']).columns
                    if c != 'id']
categorical_cols = [c for c in train_df.select_dtypes(include='object').columns
                    if c != 'health_condition']

# --- Numerical: impute with MEDIAN (robust to outliers) ---
imputation_medians = {}
print('Numerical imputation (Median):')
for col in numerical_cols:
    if train_df[col].isnull().sum() > 0:
        median_val = train_df[col].median()
        imputation_medians[col] = median_val
        train_df[col].fillna(median_val, inplace=True)
        test_df[col].fillna(median_val, inplace=True)
        print(f'   {col}: filled with median = {median_val:.4f}')

# --- Categorical: impute with MODE (most frequent value) ---
imputation_modes = {}
print('\nCategorical imputation (Mode):')
for col in categorical_cols:
    if train_df[col].isnull().sum() > 0:
        mode_val = train_df[col].mode()[0]
        imputation_modes[col] = mode_val
        train_df[col].fillna(mode_val, inplace=True)
        test_df[col].fillna(mode_val, inplace=True)
        print(f'   {col}: filled with mode = "{mode_val}"')

# Verify
remaining_train = train_df.isnull().sum().sum()
remaining_test  = test_df.isnull().sum().sum()
print(f'\n✅ Remaining missing — Train: {remaining_train} | Test: {remaining_test}')

Numerical imputation (Median):
   sleep_duration: filled with median = 6.9900
   heart_rate: filled with median = 75.1000
   bmi: filled with median = 22.9900
   calorie_expenditure: filled with median = 2241.0000
   step_count: filled with median = 8856.0000
   exercise_duration: filled with median = 39.4000
   water_intake: filled with median = 2.1700

Categorical imputation (Mode):


   diet_type: filled with mode = "veg"


   stress_level: filled with mode = "medium"
   sleep_quality: filled with mode = "average"


   physical_activity_level: filled with mode = "moderate"


   smoking_alcohol: filled with mode = "yes"
   gender: filled with mode = "male"



✅ Remaining missing — Train: 0 | Test: 0


## 4. Standardise Categorical Values

**Why:** Sometimes the same category appears as 'Male', 'male', 'MALE' — this creates false categories. We standardise to lowercase and strip whitespace.

In [5]:
# ============================================================
# SECTION 4: Standardise Categorical Strings
# ============================================================

all_cat_cols      = train_df.select_dtypes(include='object').columns.tolist()
test_cat_cols     = test_df.select_dtypes(include='object').columns.tolist()  # excludes health_condition

for col in all_cat_cols:
    # Convert to lowercase and strip whitespace
    train_df[col] = train_df[col].astype(str).str.lower().str.strip()

for col in test_cat_cols:
    test_df[col]  = test_df[col].astype(str).str.lower().str.strip()

print('Categorical columns standardised (lowercase + stripped whitespace)')
print('\nUnique values after standardisation:')
for col in all_cat_cols:
    print(f'  {col}: {list(train_df[col].unique())}')


Categorical columns standardised (lowercase + stripped whitespace)

Unique values after standardisation:
  health_condition: ['unhealthy', 'at-risk', 'fit']
  diet_type: ['veg', 'non-veg', 'balanced']
  stress_level: ['high', 'low', 'medium']
  sleep_quality: ['average', 'poor', 'good']
  physical_activity_level: ['sedentary', 'moderate', 'active']
  smoking_alcohol: ['yes', 'occasional', 'no']
  gender: ['female', 'other', 'male']


## 5. Post-Cleaning Snapshot and Verification

In [6]:
# ============================================================
# SECTION 5: Post-Cleaning Snapshot
# ============================================================

print('=' * 60)
print('  POST-CLEANING STATE')
print('=' * 60)
snapshot(train_df, 'Train')
snapshot(test_df,  'Test ')
print('=' * 60)

# Quick verification
assert train_df.isnull().sum().sum() == 0, '❌ Still has missing values in train!'
assert test_df.isnull().sum().sum() == 0,  '❌ Still has missing values in test!'
print('\n✅ Assertion passed: Zero missing values in both train and test')

  POST-CLEANING STATE


  [Train]  Rows: 690,088  |  Missing: 0  |  Duplicates: 0
  [Test ]  Rows: 295,753  |  Missing: 0  |  Duplicates: 0



✅ Assertion passed: Zero missing values in both train and test


## 6. Save Cleaned Data

**Why:** We save the cleaned datasets to `data/processed/` so the next notebooks can load the clean version — not re-run cleaning every time.

In [7]:
# ============================================================
# SECTION 6: Save Cleaned Data
# ============================================================

train_df.to_csv(PROC_DATA / 'train_cleaned.csv', index=False)
test_df.to_csv(PROC_DATA  / 'test_cleaned.csv',  index=False)

print('=' * 60)
print('  PHASE 4 COMPLETE — Data Cleaning')
print('=' * 60)
print(f'  ✅ Saved: data/processed/train_cleaned.csv  ({len(train_df):,} rows)')
print(f'  ✅ Saved: data/processed/test_cleaned.csv   ({len(test_df):,} rows)')
print('=' * 60)
print('  ✅ Ready for Phase 5: Feature Engineering')
print('=' * 60)

  PHASE 4 COMPLETE — Data Cleaning
  ✅ Saved: data/processed/train_cleaned.csv  (690,088 rows)
  ✅ Saved: data/processed/test_cleaned.csv   (295,753 rows)
  ✅ Ready for Phase 5: Feature Engineering
